# Example run of Gradient Descent Algorithm on Forward Model.

## Prepare Notebook.

### Flags for Autoreload.

In [1]:
AUTORELOAD = True
IS_EXT_LOADED = False

### Autoreload Environment.

In [2]:
if AUTORELOAD:
    if not IS_EXT_LOADED:
        %load_ext autoreload
        IS_EXT_LOADED = True
    assert IS_EXT_LOADED
    %autoreload 2

### Import Libraries.

In [3]:
from collections.abc import Collection
from datetime import datetime
from functools import partial
import logging
import os
import sys
import traceback
from typing import Callable
import uuid

import hydra
import matplotlib.pyplot as plt
import numpy as np
from omegaconf import DictConfig, OmegaConf
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch

from sc_exp_design.models import FlowMatching, FlowMatchingWithScore, FlowMap
from sc_exp_design.utils import set_reproducibility
from sc_exp_design.inverse import LossGuidedFlow

import scopt

### Registry for settings.

In [4]:
ROOT_DIR = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC"
NON_LINEARITIES_REGISTRY = {
    "identity": torch.nn.Identity,
    "relu": torch.nn.ReLU
}

### Configure Logging.

In [5]:
# 1. Configure the logging behavior
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout) # Ensures it goes to console
    ]
)
logger = logging.getLogger(__name__)


### Import Additional Modules.

In [6]:
# Import modules
sys.path.insert(0, os.path.join(ROOT_DIR, "shared_utils"))
from lambda_schedulers import schedulers_dict
from train_utils import resolve_omegaconf_to_dictionary
from experiment_utils import (
    create_dir,
    get_forward_model,
    query_forward_model,
    flatten_conf,
    get_transformed_data,
    get_target_dict,
    get_loss_fn,
)
from data_utils import get_protocol_tranformations
from plot_utils import plot_loss_history, plot_heatmap
from inverse_utils import loss_fn_factory, constraint_fn_factory, linear_scheduler_with_warmup


### Load Configurations.

In [7]:
with hydra.initialize(
    config_path="../../config"
):
    config = hydra.compose(
        config_name="run_inverse_constrained",
        overrides=[
            "paths=bloodplus_fm",
            "annotation=bloodplus",
            "sampling.N=10",
            "sampling.num_time_steps=20",
            "forward_model.num_time_steps=20",
            "constraints=default_all_cols",
            "sampling.mask_gradients=False"
        ]
    )
protocol_cols = config.annotation.protocol_columns
ndims = len(protocol_cols)

/tmp/ipykernel_822412/3314667551.py:1: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with hydra.initialize(


### Set Reproducibility.

In [8]:
# Set reproducibility
logger.info(f"Reproducibility set to {config.run.random_seed}")
set_reproducibility(config.run.random_seed)


[2026-05-27 12:27:54] - __main__: Reproducibility set to 42


## Define Potential.

### Load Forward Model.

In [9]:
forward_model, (
    _,
    target_prediction_model
) = get_forward_model(config, logger=logger)
logger.info(f"Forward model ready!\n{forward_model}")
forward_model.target_prediction_model.resc_model["model"].eval()
forward_model.forward_model.velocity_field.eval()

[2026-05-27 12:27:54] - __main__: Loading Perturbation Response Prediction Model From /lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/checkpoints/conditional_model/flow_matching/2026-04-02_14-16-39/eager-feather-1_FlowMatching.pkl...
[2026-05-27 12:30:52] - __main__: Model loaded!
NeuralVelocityField(
  (vf_modules): ModuleDict(
    (x_encoder): MLPBlock(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=26, out_features=512, bias=True)
          (1): ELU(alpha=1.0)
        )
        (1): Sequential(
          (0): Linear(in_features=512, out_features=512, bias=True)
          (1): ELU(alpha=1.0)
        )
        (2): Sequential(
          (0): Linear(in_features=512, out_features=1024, bias=True)
          (1): Identity()
        )
      )
    )
    (time_encoder): MLPBlock(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): ELU(alpha=1.0)
 

NeuralVelocityField(
  (vf_modules): ModuleDict(
    (x_encoder): MLPBlock(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=26, out_features=512, bias=True)
          (1): ELU(alpha=1.0)
        )
        (1): Sequential(
          (0): Linear(in_features=512, out_features=512, bias=True)
          (1): ELU(alpha=1.0)
        )
        (2): Sequential(
          (0): Linear(in_features=512, out_features=1024, bias=True)
          (1): Identity()
        )
      )
    )
    (time_encoder): MLPBlock(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): ELU(alpha=1.0)
        )
        (1): Sequential(
          (0): Linear(in_features=1024, out_features=1024, bias=True)
          (1): ELU(alpha=1.0)
        )
        (2): Sequential(
          (0): Linear(in_features=1024, out_features=128, bias=True)
          (1): Identity()
        )
      )
    )
    (condition_encoder

### Fit Label Encoder for Target Variable.

In [10]:
ct_le = LabelEncoder()
ct_values = target_prediction_model.train_data.adata.obs[config.loss.cell_type_column].values
ct_le.fit(ct_values)
classes = ct_le.classes_.tolist()

### Get Loss Functions.

In [11]:
# Define loss function
logger.info(f"Preparing loss function (Cross-Entropy)...")
loss_fns = get_loss_fn(config)
logger.info(f"Loss function ready!\n{loss_fns}")


[2026-05-27 12:30:57] - __main__: Preparing loss function (Cross-Entropy)...
[2026-05-27 12:30:57] - __main__: Loss function ready!
{'cell_type': <function get_loss_fn.<locals>.<lambda> at 0x7f38f4e5e020>}


### Get Target Value.

In [12]:
# Define target for current cell type
logger.info(f"Preparing target value for cell type {config.sampling.target_cell_type}...")
target = get_target_dict(
    config,
    classes,
    config.loss.cell_type_column,
    forward_model.forward_model.device
)
logger.info(f"Target ready!\n{target}")


[2026-05-27 12:30:57] - __main__: Preparing target value for cell type HSCs...
[2026-05-27 12:31:05] - __main__: Target ready!
{'cell_type': tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0.]], device='cuda:0')}


In [13]:
target["cell_type"].shape

torch.Size([1, 24])

### Initialize Non-Linearity.

In [14]:
# Initialize non linearity
logger.info(f"Initializing non linearity {config.non_linearity.non_linearity_id}...")
non_linearity_class = NON_LINEARITIES_REGISTRY.get(config.non_linearity.non_linearity_id, None)
if non_linearity_class is None:
    msg = f"Non linearity {config.non_linearity.non_linearity_id} not valid"
    raise ValueError(msg)
non_linearity = non_linearity_class(**resolve_omegaconf_to_dictionary(config.non_linearity.non_linearity_kwargs))
logger.info(f"Non linearity initialized!\n{non_linearity}")


[2026-05-27 12:31:05] - __main__: Initializing non linearity relu...
[2026-05-27 12:31:05] - __main__: Non linearity initialized!
ReLU()


### Prepare Loss.

In [15]:
# compile loss function
logger.info(f"Compiling final loss function for current cell type...")
compute_loss, noise = loss_fn_factory(
    loss_fns,
    config,
    target,
    non_linearity,
    forward_model,
)
logger.info(f"Loss function compiled!")


[2026-05-27 12:31:05] - __main__: Compiling final loss function for current cell type...
[2026-05-27 12:31:06] - __main__: Loss function compiled!


### Prepare Constraints.

In [16]:
# compile constraints
logger.info(f"Compiling constraints for current experiment...")
compute_constraints = constraint_fn_factory(config, forward_model.forward_model.device, get_protocol_tranformations)
logger.info(f"Constraints compiled!")

[2026-05-27 12:31:06] - __main__: Compiling constraints for current experiment...
[2026-05-27 12:31:06] - __main__: Constraints compiled!


### Define Potential Object.

In [26]:
class FWDPotential(scopt.potentials.BasePotential):
    def __init__(
        self,
        target,
        loss_fn: Callable[[torch.Tensor], torch.Tensor],
        penalties: Collection[Callable[[torch.Tensor], torch.Tensor]] | None = None,
        lambda_pen = 1.0,
    ):
        super().__init__()
        self._target = target
        self._loss_fn = loss_fn
        self._penalties = penalties if penalties is not None else []
        self._lambda_pen = lambda_pen
    
    def forward(
        self,
        x: torch.Tensor
    ) -> torch.Tensor:
        loss = self._loss_fn(x)
        penalties = []
        for pen_fn in self._penalties:
            pen_val = pen_fn(x)
            penalties.append(pen_val)
        penalties = torch.stack(penalties, axis=0).sum(0)
        return loss + self._lambda_pen * penalties

potential = FWDPotential(
    target,
    compute_loss,
    compute_constraints,
    lambda_pen=1.0
)

## Optimize.

### Initialize Optimization Configurations.

In [27]:
n_steps = 10_000
lr_scheduler = scopt.schedulers.LinearLR(n_steps, 1.0, 0.0)

### Initialize Optimizer.

In [30]:
gf = scopt.methods.EuclideanGradientFlow(
    potential,
    ndims,
    lr_scheduler=lr_scheduler,
    # init_distr=init_distr,
)

### Sample.

In [31]:
x1_samples = gf.sample(10)

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 19.62 GiB of which 6.88 MiB is free. Including non-PyTorch memory, this process has 19.51 GiB memory in use. Of the allocated memory 17.76 GiB is allocated by PyTorch, and 1.52 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)